## **SECTION A: SETUP**

**Objective:** Initialize the environment by loading the necessary Python libraries and authenticating the Google Maps API key.

**Tools Used:**
- `pandas` for data manipulation.
- `googlemaps` for API requests.
- `dotenv` to securely load the API key from the `.env` file.

In [1]:
import os
import pandas as pd
import googlemaps
from dotenv import load_dotenv

# Load API Key
load_dotenv()
API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")

if not API_KEY:
    raise ValueError("❌ API Key not found. Check your .env file.")

# Start Google Maps Client
gmaps = googlemaps.Client(key=API_KEY)

print("✅ Setup complete. Ready to fetch data.")

✅ Setup complete. Ready to fetch data.


## **SECTION B: FINDING THE RESORTS**

**Objective:** Search Google Places for waterfront resorts across the Kenyan coast, lakes, and safari rivers. 

**Data Collected:** 
- `ResortName`
- `Address` & `Location` (for the map)
- `Latitude` & `Longitude`
- `GoogleRating` (the overall score)
- `PlaceID` (to fetch reviews later)

In [2]:
# The search queries covering coast, lakes, and rivers
search_queries = [
    "resorts in Diani Beach Kenya",
    "beach resorts in Mombasa Shanzu Nyali",
    "resorts in Watamu and Malindi Kenya",
    "hotels in Lamu Archipelago Shela",
    "resorts around Lake Naivasha Kenya",
    "hotels near Lake Nakuru Kenya",
    "lodges around Lake Elementaita",
    "waterfront hotels in Kisumu Lake Victoria",
    "lodges along Mara River Masai Mara",
    "river lodges Samburu Ewaso Nyiro"
]

all_resorts = {}

print("🔍 Searching for resorts...")

for query in search_queries:
    try:
        response = gmaps.places(query=query)
        for place in response.get("results", []):
            place_id = place.get("place_id")
            
            # Only add if we haven't seen it before
            if place_id and place_id not in all_resorts:
                
                # Clean up the location name
                region = "Other"
                q_lower = query.lower()
                if "diani" in q_lower: region = "Coast - Diani & South"
                elif "mombasa" in q_lower or "nyali" in q_lower: region = "Coast - Mombasa & North"
                elif "watamu" in q_lower or "malindi" in q_lower: region = "Coast - Watamu & Malindi"
                elif "lamu" in q_lower: region = "Coast - Lamu"
                elif "naivasha" in q_lower: region = "Inland Lake - Naivasha"
                elif "nakuru" in q_lower: region = "Inland Lake - Nakuru"
                elif "elementaita" in q_lower: region = "Inland Lake - Elementaita"
                elif "kisumu" in q_lower or "victoria" in q_lower: region = "Inland Lake - Victoria"
                elif "mara" in q_lower or "samburu" in q_lower or "river" in q_lower: region = "Safari River"

                all_resorts[place_id] = {
                    "ResortName": place.get("name"),
                    "Address": place.get("formatted_address"),
                    "Location": region,
                    "Latitude": place.get("geometry", {}).get("location", {}).get("lat"),
                    "Longitude": place.get("geometry", {}).get("location", {}).get("lng"),
                    "GoogleRating": place.get("rating"),
                    "PlaceID": place_id
                }
    except Exception as e:
        print(f"⚠️ Error with query '{query}': {e}")

# Convert to a DataFrame
df = pd.DataFrame(list(all_resorts.values()))

# Remove the duplicate Mara River Lodge in Nairobi (if it appears)
bad_address = "Emabakasi,Airport North Road Behind Rubis Petrol Station, MWH3+899, Nairobi, Kenya"
df = df[df["Address"] != bad_address]

print(f"✅ Found {len(df)} unique resorts.")
df.head()

🔍 Searching for resorts...
✅ Found 169 unique resorts.


,ResortName,Address,Location,Latitude,Longitude,GoogleRating,PlaceID
0,Diani Reef Beach Resort & Spa,"Diani Beach Road, Diani Beach, Kenya",Coast - Diani & South,-4.278903,39.593257,4.3,ChIJJ_7_UB5GQBgRBjFxDV0y62o
1,Swahili Beach Resort,"Ukunda, Diani Beach Road, Kwale, Kenya",Coast - Diani & South,-4.292183,39.586675,4.4,ChIJlTS5haBIQBgRTH2fzpOoC5E
2,Diamonds Leisure Beach & Golf Resort,"Diani Road, South, Coast, Kwale, Kenya",Coast - Diani & South,-4.281405,39.592070,4.4,ChIJsaY6hz1GQBgRdjvawvwqcpk
3,Coral Beach Resort,"Diani Beach, Kenya",Coast - Diani & South,-4.289194,39.589740,4.1,ChIJkag91qBHQBgRQTkD5BdAgnc
4,Southern Palms Beach Resort,"Plot LR 13430, Diani Beach Road Diani, Ukunda,...",Coast - Diani & South,-4.268264,39.597005,4.3,ChIJNeN4-kRGQBgRVxlknZWsFL8


## **SECTION C: FIRST CLEANING STEP**

**Objective:** Remove duplicates, fix invalid locations, and drop any resorts that are missing critical data (like a `GoogleRating`).

In [3]:
# 1. Remove the fake "Mara River Lodge" in Nairobi (if it snuck back in)
# We use a partial string match to catch it just in case Google changed the address format
df = df[~df["Address"].str.contains("Nairobi", case=False, na=False) | ~df["ResortName"].str.contains("Mara River Lodge", case=False, na=False)]

# 2. Drop any resort that doesn't have a Google Rating
# (You can't rank a hotel if it has no score!)
df = df.dropna(subset=["GoogleRating"])

# 3. Check the final count
print(f"✅ Cleaned dataset now has {len(df)} resorts.")
print(df["Location"].value_counts())

✅ Cleaned dataset now has 168 resorts.
Location
Safari River                 22
Coast - Diani & South        20
Coast - Watamu & Malindi     20
Coast - Lamu                 20
Inland Lake - Naivasha       20
Inland Lake - Nakuru         20
Inland Lake - Victoria       20
Inland Lake - Elementaita    19
Coast - Mombasa & North       7
Name: count, dtype: int64


## **SECTION D: FETCHING REVIEWS**

**Objective:** Use the `PlaceID` to fetch the `ReviewCount` and `ReviewText` for each resort. This text will be used to power the future AI review summary, helping travelers understand what a hotel is truly known for.

In [4]:
def get_reviews(place_id):
    try:
        # Fetch total reviews and up to 5 review texts
        details = gmaps.place(
            place_id=place_id, 
            fields=["user_ratings_total", "reviews"]
        )["result"]
        
        count = details.get("user_ratings_total", 0)
        reviews_list = details.get("reviews", [])
        
        # Join the reviews into one string, separated by a delimiter
        text = " || ".join([r.get("text", "") for r in reviews_list])
        
        return pd.Series({"ReviewCount": count, "ReviewText": text})
    except Exception:
        return pd.Series({"ReviewCount": 0, "ReviewText": ""})

print("⏳ Fetching reviews for 168 resorts. This will take a few minutes...")

# Apply to every row and merge
reviews_df = df["PlaceID"].apply(get_reviews)
df = pd.concat([df, reviews_df], axis=1)

# Clean up any line breaks in the text so Tableau doesn't break
df["ReviewText"] = df["ReviewText"].fillna("").str.replace(r"[\r\n]+", " ", regex=True)

print("✅ Reviews fetched and cleaned.")
print(df[["ResortName", "ReviewCount"]].head())

⏳ Fetching reviews for 168 resorts. This will take a few minutes...
✅ Reviews fetched and cleaned.
                             ResortName  ReviewCount
0         Diani Reef Beach Resort & Spa         2759
1                  Swahili Beach Resort         3692
2  Diamonds Leisure Beach & Golf Resort         4344
3                    Coral Beach Resort          180
4           Southern Palms Beach Resort         3310


## **SECTION E: SAVING THE CLEAN DATASET**

**Objective:** Save the final, cleaned dataset to a new CSV file. This file will be the single source of truth for the Tableau dashboard.

**Final Columns Kept:**
- `ResortName`, `Address`, `Location`, `Latitude`, `Longitude`
- `GoogleRating`, `ReviewCount`, `ReviewText`
- `PlaceID`

In [5]:
# Save to a brand new CSV file
output_path = "Data/kenya_resorts_clean.csv"
df.to_csv(output_path, index=False)

print(f"✅ Dataset saved to {output_path}")
print(f"Final Shape: {df.shape[0]} rows and {df.shape[1]} columns.")
print(f"Columns: {list(df.columns)}")

✅ Dataset saved to Data/kenya_resorts_clean.csv
Final Shape: 168 rows and 9 columns.
Columns: ['ResortName', 'Address', 'Location', 'Latitude', 'Longitude', 'GoogleRating', 'PlaceID', 'ReviewCount', 'ReviewText']


## **SECTION G: GEOGRAPHIC CLEANING**

**Objective:** The original `Location` column mixes abstract categories with real places (e.g., "Coast - Diani & South"). To make the dashboard intuitive for travelers, we are splitting this into two clean columns:
- `County`: The official Kenyan county (e.g., Kwale, Nakuru, Lamu).
- `Place`: The specific destination (e.g., Diani Beach, Naivasha, Watamu).

In [9]:
def get_clean_location(row):
    loc = str(row["Location"]).lower()
    address = str(row["Address"]).lower()
    
    # Defaults
    county = "Unknown"
    place = "Unknown"
    
    if "diani" in loc or "diani" in address:
        county = "Kwale"
        place = "Diani Beach"
    elif "mombasa" in loc:
        county = "Mombasa"
        place = "Mombasa"
    elif "watamu" in loc or "watamu" in address:
        county = "Kilifi"
        place = "Watamu"
    elif "malindi" in loc or "malindi" in address:
        county = "Kilifi"
        place = "Malindi"
    elif "lamu" in loc:
        county = "Lamu"
        place = "Lamu"
    elif "naivasha" in loc:
        county = "Nakuru"
        place = "Naivasha"
    elif "elementaita" in loc:
        county = "Nakuru"
        place = "Elementaita"
    elif "nakuru" in loc:
        county = "Nakuru"
        place = "Nakuru"
    elif "victoria" in loc or "kisumu" in address:
        county = "Kisumu"
        place = "Kisumu"
    elif "safari river" in loc:
        if "mara" in address:
            county = "Narok"
            place = "Masai Mara"
        elif "samburu" in address:
            county = "Samburu"
            place = "Samburu"
    
    return pd.Series({"County": county, "Place": place})

# Apply the function to create the new columns
clean_locs = df.apply(get_clean_location, axis=1)
df = pd.concat([df, clean_locs], axis=1)

# Drop the old messy Location column
df = df.drop(columns=["Location"])

# Save the updated clean dataset
df.to_csv("Data/kenya_resorts_clean.csv", index=False)

print("✅ Geographic columns added.")
print(df[["ResortName", "County", "Place"]].head())

✅ Geographic columns added.
                             ResortName County        Place
0         Diani Reef Beach Resort & Spa  Kwale  Diani Beach
1                  Swahili Beach Resort  Kwale  Diani Beach
2  Diamonds Leisure Beach & Golf Resort  Kwale  Diani Beach
3                    Coral Beach Resort  Kwale  Diani Beach
4           Southern Palms Beach Resort  Kwale  Diani Beach


## **SECTION H: SPATIAL JOIN WITH COUNTY SHAPEFILE**

**Objective:** Replace manual text-based location guessing with a geometric spatial join. Every resort's latitude/longitude is tested against the Kenya county polygons to assign an official `County`.

**Why:** The previous method failed for 7 resorts because their address text didn't match any keyword. A spatial join is 100% reliable because it uses coordinates, not words.

In [12]:
import geopandas as gpd
import pandas as pd

# 1. Load the county shapefile from the new folder
counties = gpd.read_file("Data/kenyan-counties/County.shp")

# 2. Check the columns
print("County file columns:")
print(list(counties.columns))
print("\nFirst 3 rows:")
print(counties.head(3))

County file columns:
['OBJECTID', 'AREA', 'PERIMETER', 'COUNTY3_', 'COUNTY3_ID', 'COUNTY', 'Shape_Leng', 'Shape_Area', 'geometry']

First 3 rows:
   OBJECTID   AREA  PERIMETER  COUNTY3_  COUNTY3_ID    COUNTY  Shape_Leng  \
0         1  5.677     15.047       2.0         1.0   Turkana   15.046838   
1         2  6.177     11.974       3.0         2.0  Marsabit   11.974165   
2         3  2.117      7.355       4.0         3.0   Mandera    7.355154   

   Shape_Area                                           geometry  
0    5.676985  POLYGON ((35.79593 5.34449, 35.79659 5.34468, ...  
1    6.176831  POLYGON ((36.05061 4.45622, 36.23184 4.45124, ...  
2    2.117196  POLYGON ((41.62133 3.97673, 41.62272 3.9786, 4...  


## **SECTION I: SPATIAL JOIN (FIXING UNKNOWNS)**

**Objective:** Replace the manual text-based `County` column with a reliable spatial join. We will convert each resort's `Latitude` and `Longitude` into a geometric point and check which county polygon it falls inside.

In [13]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# 1. Load the clean dataset
df = pd.read_csv("Data/kenya_resorts_clean.csv")

# 2. Load the counties shapefile and ensure it uses standard WGS84 coordinates
counties = gpd.read_file("Data/kenyan-counties/County.shp")
counties = counties.to_crs(epsg=4326)

# 3. Create Point geometry for each resort
geometry = [Point(xy) for xy in zip(df["Longitude"], df["Latitude"])]
resorts_gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

# 4. Perform the spatial join
joined = gpd.sjoin(resorts_gdf, counties[['COUNTY', 'geometry']], how="left", predicate="intersects")

# 5. Update the main dataframe
# The shapefile uses 'COUNTY' in uppercase, so we drop the old manual 'County' and use the official one
df['County'] = joined['COUNTY']

# Check if any resorts failed the join (e.g., they are slightly offshore)
null_count = df['County'].isna().sum()
print(f"Resorts without a spatial county: {null_count}")

# If there are any nulls, fill them with "Unknown"
df['County'] = df['County'].fillna("Unknown")

# 6. Save the updated dataset
df.to_csv("Data/kenya_resorts_clean.csv", index=False)

print("✅ Spatial join complete. Final county counts:")
print(df["County"].value_counts())

Resorts without a spatial county: 31
✅ Spatial join complete. Final county counts:
County
Nakuru     59
Unknown    31
Kisumu     20
Narok      19
Lamu       13
Kwale      11
Kilifi      8
Mombasa     4
Kajiado     2
Isiolo      1
Name: count, dtype: int64


In [14]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# 1. Reload the clean dataset
df = pd.read_csv("Data/kenya_resorts_clean.csv")

# 2. Reload the shapefile
counties = gpd.read_file("Data/kenyan-counties/County.shp")
counties = counties.to_crs(epsg=4326)

# 3. Recreate the points
geometry = [Point(xy) for xy in zip(df["Longitude"], df["Latitude"])]
resorts_gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

# 4. Perform the standard join (intersects)
joined = gpd.sjoin(resorts_gdf, counties[['COUNTY', 'geometry']], how="left", predicate="intersects")

# 5. Find the ones that failed (the 31 unknowns)
unmatched_mask = joined['COUNTY'].isna()
unmatched_resorts = resorts_gdf[unmatched_mask]

# 6. Perform the nearest join ONLY for the unmatched ones
if not unmatched_resorts.empty:
    nearest_join = gpd.sjoin_nearest(unmatched_resorts, counties[['COUNTY', 'geometry']], how="left")
    
    # 7. Update the joined dataframe with the nearest county results
    joined.loc[unmatched_mask, 'COUNTY'] = nearest_join['COUNTY'].values

# 8. Update the main dataframe
df['County'] = joined['COUNTY']

# 9. Final check
print(f"Resorts without a county after Nearest Join: {df['County'].isna().sum()}")
print("\nFinal County counts:")
print(df["County"].value_counts())

# 10. Save the updated dataset
df.to_csv("Data/kenya_resorts_clean.csv", index=False)
print("✅ Saved to Data/kenya_resorts_clean.csv")

Resorts without a county after Nearest Join: 0

Final County counts:
County
Nakuru     59
Kwale      20
Kilifi     20
Lamu       20
Kisumu     20
Narok      19
Mombasa     7
Kajiado     2
Isiolo      1
Name: count, dtype: int64
✅ Saved to Data/kenya_resorts_clean.csv


c:\Users\User\desktop\GenData-Capstone-Project\venv\Lib\site-packages\geopandas\array.py:438: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


## **SECTION J: EXTRACTING STANDOUT THEMES**

**Objective:** Scan the `ReviewText` to find the most frequently mentioned theme (e.g., Views, Service, Food) for each resort. This creates a `StandoutTheme` column to show the user *why* a resort is highly rated.

In [3]:
import pandas as pd
import re
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# 1. Load your dataset
df = pd.read_csv("Data/kenya_resorts_clean.csv")

# 2. Initialize VADER
analyzer = SentimentIntensityAnalyzer()

# 3. Define themes
themes = {
    "Views & Scenery": ["view", "scenery", "beach", "ocean", "lake", "sunset"],
    "Service": ["staff", "service", "friendly", "hospitality", "welcoming"],
    "Food": ["food", "breakfast", "dinner", "meal", "buffet", "restaurant"],
    "Rooms": ["room", "bed", "bathroom", "shower", "suite"],
    "Amenities": ["pool", "spa", "wifi", "internet", "gym", "bar"]
}

def get_aspect_sentiment_theme(text):
    if pd.isna(text) or text == "":
        return "General"
    
    # Split by punctuation AND conjunctions (but, however, although) to isolate clauses
    clauses = re.split(r'[.!?]|,\s*but\s*|\s+but\s+|;\s*|,\s*however\s*', str(text), flags=re.IGNORECASE)
    
    theme_scores = {theme: 0 for theme in themes}
    
    for clause in clauses:
        if not clause.strip():
            continue
            
        clause_lower = clause.lower()
        
        for theme, keywords in themes.items():
            # If the clause mentions the theme
            if any(re.search(r'\b' + kw + r'\b', clause_lower) for kw in keywords):
                sentiment = analyzer.polarity_scores(clause)['compound']
                
                # ONLY count it if the clause is positive
                if sentiment > 0.05:
                    theme_scores[theme] += 1
                    
    best_theme = max(theme_scores, key=theme_scores.get)
    
    if theme_scores[best_theme] == 0:
        return "General"
        
    return best_theme

print("Running smarter sentiment analysis...")
df["StandoutTheme"] = df["ReviewText"].apply(get_aspect_sentiment_theme)

# Save the updated dataset
df.to_csv("Data/kenya_resorts_clean.csv", index=False)

print("Done! Dataset updated.")
print(df["StandoutTheme"].value_counts())

Running smarter sentiment analysis...
Done! Dataset updated.
StandoutTheme
Service            105
Views & Scenery     34
Food                21
Rooms                3
General              3
Amenities            2
Name: count, dtype: int64


In [4]:
# Let's verify the logic.
# We will look at a resort that has mixed reviews, and a resort that has terrible reviews.

# 1. Diani Reef Beach Resort & Spa has a mix of glowing reviews and one terrible review.
# 2. Diamond Luxury Homes watamu has mostly terrible reviews.

verify_resorts = ["Diani Reef Beach Resort & Spa", "Diamond Luxury Homes watamu", "Swahili Beach Resort"]

for resort in verify_resorts:
    row = df[df["ResortName"] == resort]
    if not row.empty:
        print(f"\n{'='*50}")
        print(f"RESORT: {resort}")
        print(f"ASSIGNED THEME: {row['StandoutTheme'].values[0]}")
        print(f"{'-'*50}")
        # Print the first 300 characters of the review text so it fits on screen
        print("SAMPLE REVIEW TEXT:")
        print(str(row['ReviewText'].values[0])[:300] + "...\n")


RESORT: Diani Reef Beach Resort & Spa
ASSIGNED THEME: Service
--------------------------------------------------
SAMPLE REVIEW TEXT:
I absolutely loved the place. Right from the reservation which was done well by Mrs Sandra to the service everything was done well. We initially booked a garden view but upon request we were switched to sea view which was so thoughtful. The rooms are clean and the buffet had a variety of options wit...


RESORT: Diamond Luxury Homes watamu
ASSIGNED THEME: Service
--------------------------------------------------
SAMPLE REVIEW TEXT:
Do not rent this place!! The property is in very poor condition. We had repeated power and water outages, non-working air conditioning, and a serious problem with rodents and cockroaches in both the bedrooms and bathrooms. It made the stay completely unacceptable. Security is also a major issue. The...


RESORT: Swahili Beach Resort
ASSIGNED THEME: Service
--------------------------------------------------
SAMPLE REVIEW TEXT:

## **SECTION K: FETCHING REAL PRICES (APIFY)**

**Objective:** To fulfill Dimension 4 (Pricing & Value), we need real pricing data. The standard Google Places API does not provide base prices, so we had to integrate a third-party web scraper.

**The Solution:** We used the Apify Google Hotels Scraper to fetch the base price (in USD), hotel class (e.g., "4-star hotel"), and property category for our resorts. 

**Methodology & Challenges:** Apify's free tier limits usage to $5.00 per account. To scrape the full dataset of 168 resorts, we divided the scraping process into multiple batches and used separate free accounts to bypass the limit. The scraped data was then merged into our master dataset. A `BasePrice_KES` column was also created using a standard exchange rate (1 USD ≈ 150 KES) for local context.

**Limitations:** While we scraped all 168 resorts, not every resort returned a price. Some smaller guesthouses are not listed on Google Hotels for the selected dates, and some searches returned 0 results. This is a common real-world data gap that is documented in our final dataset.

In [2]:
import os
import pandas as pd
from apify_client import ApifyClient
from dotenv import load_dotenv
import time

# 1. Load the Apify token from .env
load_dotenv()
APIFY_TOKEN = os.getenv("APIFY_API_TOKEN")

if not APIFY_TOKEN:
    raise ValueError("❌ APIFY_API_TOKEN not found in .env file. Check Step 2!")

client = ApifyClient(APIFY_TOKEN)

# 2. Load the dataset
df = pd.read_csv("Data/kenya_resorts_clean.csv")

# 3. Set the limit (Change to None to run all 168 later)
TEST_LIMIT = 5 
df_to_scrape = df.head(TEST_LIMIT)

print(f"⏳ Scraping prices and amenities for {len(df_to_scrape)} resorts. This will take a few minutes...")

results = []

for index, row in df_to_scrape.iterrows():
    resort_name = row["ResortName"]
    print(f"Scraping: {resort_name}...")
    
    # We use a dummy future date to get a standard base price
    run_input = {
        "destination": f"{resort_name}, Kenya",
        "checkIn": "2026-12-01",  
        "checkOut": "2026-12-02",
        "adults": 2,
        "currency": "USD",
        "maxResultsPerSearch": 1 
    }
    
    try:
        # Run the Actor
        run = client.actor("inexhaustible_glass/google-hotels-scraper").call(run_input=run_input)
        
        # Fetch the results
        item_found = False
        
        # THE FIX IS HERE: changed run["defaultDatasetId"] to run.default_dataset_id
        for item in client.dataset(run.default_dataset_id).iterate_items():
            results.append({
                "ResortName": resort_name,
                "ScrapedPrice": item.get("price"),
                "ScrapedAmenities": ", ".join(item.get("amenities", []))
            })
            item_found = True
            break # Just take the first result
            
        if not item_found:
            results.append({
                "ResortName": resort_name,
                "ScrapedPrice": None,
                "ScrapedAmenities": None
            })
            
        # Pause for a second to respect Apify's rate limits
        time.sleep(2)
        
    except Exception as e:
        print(f"⚠️ Error scraping {resort_name}: {e}")
        results.append({
            "ResortName": resort_name,
            "ScrapedPrice": None,
            "ScrapedAmenities": None
        })

# 4. Show the results
apify_df = pd.DataFrame(results)
print("\n✅ Scraping complete. Here is what we got:")
print(apify_df.to_string())

⏳ Scraping prices and amenities for 5 resorts. This will take a few minutes...
Scraping: Diani Reef Beach Resort & Spa...


[apify.google-hotels-scraper runId:dNjxKqPjhBxDQ7kIz] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:dNjxKqPjhBxDQ7kIz] -> 2026-11-14T15:51:44.581Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:dNjxKqPjhBxDQ7kIz] -> 2026-11-14T15:51:44.583Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:dNjxKqPjhBxDQ7kIz] -> 2026-11-14T15:51:44.630Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:dNjxKqPjhBxDQ7kIz] -> 2026-11-14T15:51:44.630Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:dNjxKqPjhBxDQ7kIz] -> 2026-11-14T15:51:45.419Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:dNjxKqPjhBxDQ7kIz] -> 2026-11-14T15:51:45.420Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

Scraping: Swahili Beach Resort...


[apify.google-hotels-scraper runId:R8swAijlWXFvNWkRj] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:R8swAijlWXFvNWkRj] -> 2026-11-14T15:51:57.552Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:R8swAijlWXFvNWkRj] -> 2026-11-14T15:51:57.555Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:R8swAijlWXFvNWkRj] -> 2026-11-14T15:51:57.609Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:R8swAijlWXFvNWkRj] -> 2026-11-14T15:51:57.610Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:R8swAijlWXFvNWkRj] -> 2026-11-14T15:51:58.465Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:R8swAijlWXFvNWkRj] -> 2026-11-14T15:51:58.466Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

Scraping: Diamonds Leisure Beach & Golf Resort...


[apify.google-hotels-scraper runId:XbWMJKrffNmATbqJc] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:XbWMJKrffNmATbqJc] -> 2026-11-14T15:52:10.499Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:XbWMJKrffNmATbqJc] -> 2026-11-14T15:52:10.501Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:XbWMJKrffNmATbqJc] -> 2026-11-14T15:52:10.550Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:XbWMJKrffNmATbqJc] -> 2026-11-14T15:52:10.551Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:XbWMJKrffNmATbqJc] -> 2026-11-14T15:52:11.335Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:XbWMJKrffNmATbqJc] -> 2026-11-14T15:52:11.336Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

Scraping: Coral Beach Resort...


[apify.google-hotels-scraper runId:h6ti1JPSHeF4TEc1Y] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:h6ti1JPSHeF4TEc1Y] -> 2026-11-14T15:52:23.366Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:h6ti1JPSHeF4TEc1Y] -> 2026-11-14T15:52:23.368Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:h6ti1JPSHeF4TEc1Y] -> 2026-11-14T15:52:23.424Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:h6ti1JPSHeF4TEc1Y] -> 2026-11-14T15:52:23.424Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:h6ti1JPSHeF4TEc1Y] -> 2026-11-14T15:52:24.273Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:h6ti1JPSHeF4TEc1Y] -> 2026-11-14T15:52:24.274Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

Scraping: Southern Palms Beach Resort...


[apify.google-hotels-scraper runId:2XBmdP6bX8lbindot] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:2XBmdP6bX8lbindot] -> 2026-11-14T15:52:36.200Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:2XBmdP6bX8lbindot] -> 2026-11-14T15:52:36.203Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:2XBmdP6bX8lbindot] -> 2026-11-14T15:52:36.257Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:2XBmdP6bX8lbindot] -> 2026-11-14T15:52:36.257Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:2XBmdP6bX8lbindot] -> 2026-11-14T15:52:37.113Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:2XBmdP6bX8lbindot] -> 2026-11-14T15:52:37.114Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin


✅ Scraping complete. Here is what we got:
                             ResortName  ScrapedPrice ScrapedAmenities
0         Diani Reef Beach Resort & Spa         150.0                 
1                  Swahili Beach Resort         267.0                 
2  Diamonds Leisure Beach & Golf Resort         180.0                 
3                    Coral Beach Resort           NaN              NaN
4           Southern Palms Beach Resort         199.0                 


In [3]:
import os
from apify_client import ApifyClient
from dotenv import load_dotenv

load_dotenv()
client = ApifyClient(os.getenv("APIFY_API_TOKEN"))

print("Scraping just Diani Reef to inspect the raw keys...")
run = client.actor("inexhaustible_glass/google-hotels-scraper").call(run_input={
    "destination": "Diani Reef Beach Resort & Spa, Kenya",
    "checkIn": "2026-12-01",
    "checkOut": "2026-12-02",
    "adults": 2,
    "currency": "USD",
    "maxResultsPerSearch": 1 
})

for item in client.dataset(run.default_dataset_id).iterate_items():
    # Print all the keys (column names) that Apify returned
    print("\n--- AVAILABLE KEYS ---")
    for key in item.keys():
        print(f"• {key}")
        
    # Look specifically for anything related to 'amenity'
    print("\n--- RAW AMENITY DATA ---")
    for key, value in item.items():
        if "amenit" in key.lower() or "feature" in key.lower():
            print(f"{key}: {value}")
    break

Scraping just Diani Reef to inspect the raw keys...


[apify.google-hotels-scraper runId:JimSUv2fFEGwnk9hD] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:JimSUv2fFEGwnk9hD] -> 2026-11-14T15:53:19.222Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:JimSUv2fFEGwnk9hD] -> 2026-11-14T15:53:19.224Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:JimSUv2fFEGwnk9hD] -> 2026-11-14T15:53:19.274Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:JimSUv2fFEGwnk9hD] -> 2026-11-14T15:53:19.276Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:JimSUv2fFEGwnk9hD] -> 2026-11-14T15:53:20.054Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:JimSUv2fFEGwnk9hD] -> 2026-11-14T15:53:20.056Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin


--- AVAILABLE KEYS ---
• hotel_id
• name
• price
• total_price
• currency
• rating
• review_count
• hotel_class
• hotel_class_label
• latitude
• longitude
• address
• image_url
• distance
• booking_token
• price_token
• place_id
• entity_id
• providers
• success
• error
• search_index
• destination_query
• check_in
• check_out
• property_category

--- RAW AMENITY DATA ---


In [4]:
import os
import pandas as pd
from apify_client import ApifyClient
from dotenv import load_dotenv
import time

# 1. Load the Apify token
load_dotenv()
APIFY_TOKEN = os.getenv("APIFY_API_TOKEN")
client = ApifyClient(APIFY_TOKEN)

# 2. Load the dataset
df = pd.read_csv("Data/kenya_resorts_clean.csv")

print(f"⏳ Scraping PRICES for all {len(df)} resorts. This will take 10-15 minutes...")

results = []

for index, row in df.iterrows():
    resort_name = row["ResortName"]
    print(f"[{index+1}/168] Scraping: {resort_name}...")
    
    run_input = {
        "destination": f"{resort_name}, Kenya",
        "checkIn": "2026-12-01",  
        "checkOut": "2026-12-02",
        "adults": 2,
        "currency": "USD",
        "maxResultsPerSearch": 1 
    }
    
    try:
        run = client.actor("inexhaustible_glass/google-hotels-scraper").call(run_input=run_input)
        
        item_found = False
        for item in client.dataset(run.default_dataset_id).iterate_items():
            results.append({
                "ResortName": resort_name,
                "BasePrice": item.get("price"),
                "HotelClass": item.get("hotel_class_label"),
                "Category": item.get("property_category")
            })
            item_found = True
            break 
            
        if not item_found:
            results.append({"ResortName": resort_name, "BasePrice": None, "HotelClass": None, "Category": None})
            
        time.sleep(2) # Respect rate limits
        
    except Exception as e:
        print(f"⚠️ Error: {e}")
        results.append({"ResortName": resort_name, "BasePrice": None, "HotelClass": None, "Category": None})

# 3. Merge and Save
apify_df = pd.DataFrame(results)
final_df = pd.merge(df, apify_df, on="ResortName", how="left")

# Save to a NEW file so you don't lose your clean data if it crashes
final_df.to_csv("Data/kenya_resorts_master.csv", index=False)

print("\n✅ Scraping complete! Dataset saved to 'Data/kenya_resorts_master.csv'")
print("Resorts with a valid price:", final_df["BasePrice"].notna().sum())
print(final_df[["ResortName", "BasePrice", "HotelClass"]].head(10))

⏳ Scraping PRICES for all 168 resorts. This will take 10-15 minutes...
[1/168] Scraping: Diani Reef Beach Resort & Spa...


[apify.google-hotels-scraper runId:OmHIT2yl1IyZi4LVE] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:OmHIT2yl1IyZi4LVE] -> 2026-11-14T15:54:18.253Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:OmHIT2yl1IyZi4LVE] -> 2026-11-14T15:54:19.362Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:OmHIT2yl1IyZi4LVE] -> 2026-11-14T15:54:19.422Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:OmHIT2yl1IyZi4LVE] -> 2026-11-14T15:54:19.424Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:OmHIT2yl1IyZi4LVE] -> 2026-11-14T15:54:20.244Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:OmHIT2yl1IyZi4LVE] -> 2026-11-14T15:54:20.245Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

[2/168] Scraping: Swahili Beach Resort...


[apify.google-hotels-scraper runId:iVuUVyAaodxyWGvqc] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:iVuUVyAaodxyWGvqc] -> 2026-11-14T15:54:32.510Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:iVuUVyAaodxyWGvqc] -> 2026-11-14T15:54:32.513Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:iVuUVyAaodxyWGvqc] -> 2026-11-14T15:54:32.563Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:iVuUVyAaodxyWGvqc] -> 2026-11-14T15:54:32.563Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:iVuUVyAaodxyWGvqc] -> 2026-11-14T15:54:33.346Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:iVuUVyAaodxyWGvqc] -> 2026-11-14T15:54:33.347Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

[3/168] Scraping: Diamonds Leisure Beach & Golf Resort...


[apify.google-hotels-scraper runId:KvJqhZwD123Cyxzl6] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:KvJqhZwD123Cyxzl6] -> 2026-11-14T15:54:44.483Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:KvJqhZwD123Cyxzl6] -> 2026-11-14T15:54:44.486Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:KvJqhZwD123Cyxzl6] -> 2026-11-14T15:54:44.536Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:KvJqhZwD123Cyxzl6] -> 2026-11-14T15:54:44.537Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:KvJqhZwD123Cyxzl6] -> 2026-11-14T15:54:45.334Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:KvJqhZwD123Cyxzl6] -> 2026-11-14T15:54:45.336Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

... (87 outputs left out)

[apify.google-hotels-scraper runId:vrlFaN3ZXb1LmJ8BA] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:vrlFaN3ZXb1LmJ8BA] -> 2026-11-14T16:17:18.415Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:vrlFaN3ZXb1LmJ8BA] -> 2026-11-14T16:17:18.421Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:vrlFaN3ZXb1LmJ8BA] -> 2026-11-14T16:17:18.941Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:vrlFaN3ZXb1LmJ8BA] -> 2026-11-14T16:17:18.942Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:vrlFaN3ZXb1LmJ8BA] -> 2026-11-14T16:17:19.877Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:vrlFaN3ZXb1LmJ8BA] -> 2026-11-14T16:17:19.877Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

[48/168] Scraping: The Majlis Hotel Lamu...


[apify.google-hotels-scraper runId:DFm15focffkfcAL92] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:DFm15focffkfcAL92] -> 2026-11-14T16:17:32.805Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:DFm15focffkfcAL92] -> 2026-11-14T16:17:32.807Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:DFm15focffkfcAL92] -> 2026-11-14T16:17:32.858Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:DFm15focffkfcAL92] -> 2026-11-14T16:17:32.859Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:DFm15focffkfcAL92] -> 2026-11-14T16:17:34.188Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:DFm15focffkfcAL92] -> 2026-11-14T16:17:34.189Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

[49/168] Scraping: Shela Bliss...
⚠️ Error: POST /v2/actors/inexhaustible_glass~google-hotels-scraper/runs: You've used the $5.00 of monthly usage included in the Free plan, so Actor runs and other Apify services are blocked until your monthly usage cycle resets. GET https://api.apify.com/v2/users/me/limits shows when (monthlyUsageCycle.endAt). The Free plan limit can't be raised. To continue now, upgrade at https://console.apify.com/billing/subscription.
[50/168] Scraping: Wiyoni Royals Lamu...
⚠️ Error: POST /v2/actors/inexhaustible_glass~google-hotels-scraper/runs: You've used the $5.00 of monthly usage included in the Free plan, so Actor runs and other Apify services are blocked until your monthly usage cycle resets. GET https://api.apify.com/v2/users/me/limits shows when (monthlyUsageCycle.endAt). The Free plan limit can't be raised. To continue now, upgrade at https://console.apify.com/billing/subscription.
[51/168] Scraping: Banana House And Wellness Centre...
⚠️ Error: POST /v2

In [5]:
import pandas as pd

# 1. Load your clean dataset
df = pd.read_csv("Data/kenya_resorts_clean.csv")

# 2. Merge the Apify data (from your memory) 
# We use 'how="left"' so the 120 resorts that failed just get NaN (empty) prices
final_df = pd.merge(df, apify_df, on="ResortName", how="left")

# 3. Add the KES conversion (Approx 150 KES to 1 USD)
final_df["BasePrice_KES"] = final_df["BasePrice"] * 150

# 4. Overwrite the EXACT SAME file Tableau is using
final_df.to_csv("Data/kenya_resorts_clean.csv", index=False)

print("✅ Successfully saved prices to kenya_resorts_clean.csv")
print("Resorts with prices:", final_df["BasePrice"].notna().sum())
print(final_df[["ResortName", "BasePrice", "BasePrice_KES", "HotelClass"]].head(10))

✅ Successfully saved prices to kenya_resorts_clean.csv
Resorts with prices: 26
                             ResortName  BasePrice  BasePrice_KES  \
0         Diani Reef Beach Resort & Spa      150.0        22500.0   
1                  Swahili Beach Resort      267.0        40050.0   
2  Diamonds Leisure Beach & Golf Resort      180.0        27000.0   
3                    Coral Beach Resort        NaN            NaN   
4           Southern Palms Beach Resort      199.0        29850.0   
5   Jacaranda Indian Ocean Beach Resort      112.0        16800.0   
6                  Kaskazi Beach Resort      106.0        15900.0   
7         The Villa Luxury Suites Hotel       86.0        12900.0   
8                           Tombo House       98.0        14700.0   
9              Bahari Dhow Beach Villas       95.0        14250.0   

     HotelClass  
0  5-star hotel  
1  5-star hotel  
2  5-star hotel  
3           NaN  
4  4-star hotel  
5  4-star hotel  
6  4-star hotel  
7  4-star hotel  

In [6]:
import os
import pandas as pd
from apify_client import ApifyClient
from dotenv import load_dotenv
import time

# 1. Load the NEW Apify token
load_dotenv()
APIFY_TOKEN = os.getenv("APIFY_API_TOKEN")
client = ApifyClient(APIFY_TOKEN)

# 2. Load the dataset that already has the first 48 prices
df = pd.read_csv("Data/kenya_resorts_clean.csv")

# 3. Skip the first 48 resorts and scrape the rest
df_to_scrape = df.iloc[48:] 
print(f"⏳ Scraping the remaining {len(df_to_scrape)} resorts. This will take 5-10 minutes...")

results = []

for index, row in df_to_scrape.iterrows():
    resort_name = row["ResortName"]
    print(f"[{index+1}/168] Scraping: {resort_name}...")
    
    run_input = {
        "destination": f"{resort_name}, Kenya",
        "checkIn": "2026-12-01",  
        "checkOut": "2026-12-02",
        "adults": 2,
        "currency": "USD",
        "maxResultsPerSearch": 1 
    }
    
    try:
        run = client.actor("inexhaustible_glass/google-hotels-scraper").call(run_input=run_input)
        
        item_found = False
        for item in client.dataset(run.default_dataset_id).iterate_items():
            results.append({
                "ResortName": resort_name,
                "BasePrice_new": item.get("price"),
                "HotelClass_new": item.get("hotel_class_label"),
                "Category_new": item.get("property_category")
            })
            item_found = True
            break 
            
        if not item_found:
            results.append({"ResortName": resort_name, "BasePrice_new": None, "HotelClass_new": None, "Category_new": None})
            
        time.sleep(2) # Respect rate limits
        
    except Exception as e:
        print(f"⚠️ Error: {e}")
        results.append({"ResortName": resort_name, "BasePrice_new": None, "HotelClass_new": None, "Category_new": None})

# 4. Merge the new data with the old data
new_apify_df = pd.DataFrame(results)
final_df = pd.merge(df, new_apify_df, on="ResortName", how="left")

# 5. Update the main columns with the new data where it's missing
final_df["BasePrice"] = final_df["BasePrice"].fillna(final_df["BasePrice_new"])
final_df["HotelClass"] = final_df["HotelClass"].fillna(final_df["HotelClass_new"])
final_df["Category"] = final_df["Category"].fillna(final_df["Category_new"])

# 6. Drop the temporary "_new" columns
final_df = final_df.drop(columns=["BasePrice_new", "HotelClass_new", "Category_new"])

# 7. Recalculate KES for the whole dataset
final_df["BasePrice_KES"] = final_df["BasePrice"] * 150

# 8. Save it back to the SAME file Tableau uses
final_df.to_csv("Data/kenya_resorts_clean.csv", index=False)

print("\n✅ Scraping complete! Dataset fully updated.")
print("Total Resorts with prices:", final_df["BasePrice"].notna().sum())

⏳ Scraping the remaining 120 resorts. This will take 5-10 minutes...
[49/168] Scraping: Shela Bliss...
⚠️ Error: POST /v2/actors/inexhaustible_glass~google-hotels-scraper/runs: You've used the $5.00 of monthly usage included in the Free plan, so Actor runs and other Apify services are blocked until your monthly usage cycle resets. GET https://api.apify.com/v2/users/me/limits shows when (monthlyUsageCycle.endAt). The Free plan limit can't be raised. To continue now, upgrade at https://console.apify.com/billing/subscription.
[50/168] Scraping: Wiyoni Royals Lamu...
⚠️ Error: POST /v2/actors/inexhaustible_glass~google-hotels-scraper/runs: You've used the $5.00 of monthly usage included in the Free plan, so Actor runs and other Apify services are blocked until your monthly usage cycle resets. GET https://api.apify.com/v2/users/me/limits shows when (monthlyUsageCycle.endAt). The Free plan limit can't be raised. To continue now, upgrade at https://console.apify.com/billing/subscription.
[51/

C:\Users\User\AppData\Local\Temp\ipykernel_3436\1593249106.py:62: Pandas4Warning: 'ndarray' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  final_df["BasePrice"] = final_df["BasePrice"].fillna(final_df["BasePrice_new"])


In [7]:
import os
import pandas as pd
from apify_client import ApifyClient
from dotenv import load_dotenv
import time

# 1. FORCE reload the new token
load_dotenv(override=True) 
APIFY_TOKEN = os.getenv("APIFY_API_TOKEN")
client = ApifyClient(APIFY_TOKEN)

# 2. Load dataset
df = pd.read_csv("Data/kenya_resorts_clean.csv")

# 3. Find ONLY the resorts that don't have a price yet
missing_df = df[df["BasePrice"].isna()]
print(f"⏳ Found {len(missing_df)} resorts without prices. Scraping them now...")

results = []

for index, row in missing_df.iterrows():
    resort_name = row["ResortName"]
    print(f"Scraping: {resort_name}...")
    
    run_input = {
        "destination": f"{resort_name}, Kenya",
        "checkIn": "2026-12-01",  
        "checkOut": "2026-12-02",
        "adults": 2,
        "currency": "USD",
        "maxResultsPerSearch": 1 
    }
    
    try:
        run = client.actor("inexhaustible_glass/google-hotels-scraper").call(run_input=run_input)
        
        item_found = False
        for item in client.dataset(run.default_dataset_id).iterate_items():
            price = item.get("price")
            results.append({
                "ResortName": resort_name,
                "BasePrice_new": price,
                "HotelClass_new": item.get("hotel_class_label"),
                "BasePrice_KES_new": price * 150 if price else None
            })
            item_found = True
            break 
            
        if not item_found:
            results.append({"ResortName": resort_name, "BasePrice_new": None, "HotelClass_new": None, "BasePrice_KES_new": None})
            
        time.sleep(2) # Respect rate limits
        
    except Exception as e:
        print(f"⚠️ Error on {resort_name}: {e}")
        results.append({"ResortName": resort_name, "BasePrice_new": None, "HotelClass_new": None, "BasePrice_KES_new": None})

# 4. Merge the new prices into the main dataframe
new_apify_df = pd.DataFrame(results)
final_df = pd.merge(df, new_apify_df, on="ResortName", how="left")

# 5. Update the main columns where the new data exists
final_df["BasePrice"] = final_df["BasePrice"].fillna(final_df["BasePrice_new"])
final_df["HotelClass"] = final_df["HotelClass"].fillna(final_df["HotelClass_new"])
final_df["BasePrice_KES"] = final_df["BasePrice_KES"].fillna(final_df["BasePrice_KES_new"])

# 6. Drop the temporary columns
final_df = final_df.drop(columns=["BasePrice_new", "HotelClass_new", "BasePrice_KES_new"])

# 7. Save to the exact same file
final_df.to_csv("Data/kenya_resorts_clean.csv", index=False)

print("\n✅ Scraping complete!")
print("Total Resorts with prices:", final_df["BasePrice"].notna().sum())

⏳ Found 142 resorts without prices. Scraping them now...
Scraping: Coral Beach Resort...


[apify.google-hotels-scraper runId:BuhjCXzUq5oteQS28] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:BuhjCXzUq5oteQS28] -> 2026-11-14T16:29:35.256Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:BuhjCXzUq5oteQS28] -> 2026-11-14T16:29:35.264Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:BuhjCXzUq5oteQS28] -> 2026-11-14T16:29:35.323Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:BuhjCXzUq5oteQS28] -> 2026-11-14T16:29:35.324Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:BuhjCXzUq5oteQS28] -> 2026-11-14T16:29:36.880Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:BuhjCXzUq5oteQS28] -> 2026-11-14T16:29:36.881Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

Scraping: Aqua Resort...


[apify.google-hotels-scraper runId:QffTNeCrFhqgBZ22E] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:QffTNeCrFhqgBZ22E] -> 2026-11-14T16:29:49.400Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:QffTNeCrFhqgBZ22E] -> 2026-11-14T16:29:49.403Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:QffTNeCrFhqgBZ22E] -> 2026-11-14T16:29:49.477Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:QffTNeCrFhqgBZ22E] -> 2026-11-14T16:29:49.478Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:QffTNeCrFhqgBZ22E] -> 2026-11-14T16:29:51.105Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:QffTNeCrFhqgBZ22E] -> 2026-11-14T16:29:51.109Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

Scraping: Alfajiri Villas...


[apify.google-hotels-scraper runId:lAOHf2dWmiKEtUYbo] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:lAOHf2dWmiKEtUYbo] -> 2026-11-14T16:30:03.541Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:lAOHf2dWmiKEtUYbo] -> 2026-11-14T16:30:03.543Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:lAOHf2dWmiKEtUYbo] -> 2026-11-14T16:30:03.596Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:lAOHf2dWmiKEtUYbo] -> 2026-11-14T16:30:03.598Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:lAOHf2dWmiKEtUYbo] -> 2026-11-14T16:30:04.419Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:lAOHf2dWmiKEtUYbo] -> 2026-11-14T16:30:04.420Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

... (117 outputs left out)

[apify.google-hotels-scraper runId:nW6JVoEPBq6LoR0sw] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:nW6JVoEPBq6LoR0sw] -> 2026-11-14T16:43:09.243Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:nW6JVoEPBq6LoR0sw] -> 2026-11-14T16:43:09.246Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:nW6JVoEPBq6LoR0sw] -> 2026-11-14T16:43:09.304Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:nW6JVoEPBq6LoR0sw] -> 2026-11-14T16:43:09.305Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:nW6JVoEPBq6LoR0sw] -> 2026-11-14T16:43:10.586Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:nW6JVoEPBq6LoR0sw] -> 2026-11-14T16:43:10.587Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

Scraping: Liaizons Hotel Nakuru...


[apify.google-hotels-scraper runId:Chufh5iWXKUMYHH6N] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:Chufh5iWXKUMYHH6N] -> 2026-11-14T16:43:22.132Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:Chufh5iWXKUMYHH6N] -> 2026-11-14T16:43:22.135Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:Chufh5iWXKUMYHH6N] -> 2026-11-14T16:43:22.192Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:Chufh5iWXKUMYHH6N] -> 2026-11-14T16:43:22.195Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:Chufh5iWXKUMYHH6N] -> 2026-11-14T16:43:23.643Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:Chufh5iWXKUMYHH6N] -> 2026-11-14T16:43:23.645Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

Scraping: Eldorado Rustic Hotel & Gardens...
⚠️ Error on Eldorado Rustic Hotel & Gardens: POST /v2/actors/inexhaustible_glass~google-hotels-scraper/runs: You've used the $5.00 of monthly usage included in the Free plan, so Actor runs and other Apify services are blocked until your monthly usage cycle resets. GET https://api.apify.com/v2/users/me/limits shows when (monthlyUsageCycle.endAt). The Free plan limit can't be raised. To continue now, upgrade at https://console.apify.com/billing/subscription.
Scraping: Lanet Matfam Resort...
⚠️ Error on Lanet Matfam Resort: POST /v2/actors/inexhaustible_glass~google-hotels-scraper/runs: You've used the $5.00 of monthly usage included in the Free plan, so Actor runs and other Apify services are blocked until your monthly usage cycle resets. GET https://api.apify.com/v2/users/me/limits shows when (monthlyUsageCycle.endAt). The Free plan limit can't be raised. To continue now, upgrade at https://console.apify.com/billing/subscription.
Scraping: Th

In [8]:
import os
import pandas as pd
from apify_client import ApifyClient
from dotenv import load_dotenv
import time

# 1. Load the NEW Apify token
load_dotenv(override=True) 
APIFY_TOKEN = os.getenv("APIFY_API_TOKEN")
client = ApifyClient(APIFY_TOKEN)

# 2. Load your main dataset
df = pd.read_csv("Data/kenya_resorts_clean.csv")

# 3. Start from index 128 (where we left off)
df_to_scrape = df.iloc[128:]
print(f"⏳ Scraping the final {len(df_to_scrape)} resorts...")

results = []

for index, row in df_to_scrape.iterrows():
    resort_name = row["ResortName"]
    print(f"[{index+1}/168] Scraping: {resort_name}...")
    
    run_input = {
        "destination": f"{resort_name}, Kenya",
        "checkIn": "2026-12-01",  
        "checkOut": "2026-12-02",
        "adults": 2,
        "currency": "USD",
        "maxResultsPerSearch": 1 
    }
    
    try:
        run = client.actor("inexhaustible_glass/google-hotels-scraper").call(run_input=run_input)
        
        item_found = False
        for item in client.dataset(run.default_dataset_id).iterate_items():
            price = item.get("price")
            results.append({
                "ResortName": resort_name,
                "BasePrice": price,
                "HotelClass": item.get("hotel_class_label"),
                "BasePrice_KES": price * 150 if price else None
            })
            item_found = True
            break 
            
        if not item_found:
            results.append({"ResortName": resort_name, "BasePrice": None, "HotelClass": None, "BasePrice_KES": None})
            
        time.sleep(2) 
        
    except Exception as e:
        print(f"⚠️ Error: {e}")
        results.append({"ResortName": resort_name, "BasePrice": None, "HotelClass": None, "BasePrice_KES": None})

# Save these new results to a separate file so we don't lose them
new_apify_df = pd.DataFrame(results)
new_apify_df.to_csv("Data/scraped_prices_part3.csv", index=False)

print("\n✅ Part 3 Scraping complete! Saved to scraped_prices_part3.csv")

⏳ Scraping the final 40 resorts...
[129/168] Scraping: Sarova Imperial Hotel, Kisumu...


[apify.google-hotels-scraper runId:5uExW8Cf40fd935jc] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:5uExW8Cf40fd935jc] -> 2026-11-14T16:51:36.850Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:5uExW8Cf40fd935jc] -> 2026-11-14T16:51:36.853Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:5uExW8Cf40fd935jc] -> 2026-11-14T16:51:36.909Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:5uExW8Cf40fd935jc] -> 2026-11-14T16:51:36.910Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:5uExW8Cf40fd935jc] -> 2026-11-14T16:51:38.033Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:5uExW8Cf40fd935jc] -> 2026-11-14T16:51:38.034Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

[130/168] Scraping: Dunga Lakeside Resort...


[apify.google-hotels-scraper runId:eOPmAMZLbcbCboimW] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:eOPmAMZLbcbCboimW] -> 2026-11-14T16:51:49.950Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:eOPmAMZLbcbCboimW] -> 2026-11-14T16:51:49.953Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:eOPmAMZLbcbCboimW] -> 2026-11-14T16:51:50.009Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:eOPmAMZLbcbCboimW] -> 2026-11-14T16:51:50.015Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:eOPmAMZLbcbCboimW] -> 2026-11-14T16:51:51.079Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:eOPmAMZLbcbCboimW] -> 2026-11-14T16:51:51.081Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

[131/168] Scraping: Kiboko Bay Resort...


[apify.google-hotels-scraper runId:xFTjq92UTXdopIdnD] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:xFTjq92UTXdopIdnD] -> 2026-11-14T16:52:02.875Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:xFTjq92UTXdopIdnD] -> 2026-11-14T16:52:02.877Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:xFTjq92UTXdopIdnD] -> 2026-11-14T16:52:02.960Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:xFTjq92UTXdopIdnD] -> 2026-11-14T16:52:02.961Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:xFTjq92UTXdopIdnD] -> 2026-11-14T16:52:03.964Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:xFTjq92UTXdopIdnD] -> 2026-11-14T16:52:03.966Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

... (71 outputs left out)

[apify.google-hotels-scraper runId:X3kTZhKgfuaan1YsP] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:X3kTZhKgfuaan1YsP] -> 2026-11-14T16:59:51.630Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:X3kTZhKgfuaan1YsP] -> 2026-11-14T16:59:51.633Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:X3kTZhKgfuaan1YsP] -> 2026-11-14T16:59:51.684Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:X3kTZhKgfuaan1YsP] -> 2026-11-14T16:59:51.686Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:X3kTZhKgfuaan1YsP] -> 2026-11-14T16:59:52.927Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:X3kTZhKgfuaan1YsP] -> 2026-11-14T16:59:52.930Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

[168/168] Scraping: Samburu Riverside Camp...


[apify.google-hotels-scraper runId:dA4YIaWSOmO0EWgm0] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:dA4YIaWSOmO0EWgm0] -> 2026-11-14T17:00:04.674Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:dA4YIaWSOmO0EWgm0] -> 2026-11-14T17:00:04.727Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:dA4YIaWSOmO0EWgm0] -> 2026-11-14T17:00:04.819Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:dA4YIaWSOmO0EWgm0] -> 2026-11-14T17:00:04.821Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:dA4YIaWSOmO0EWgm0] -> 2026-11-14T17:00:06.120Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:dA4YIaWSOmO0EWgm0] -> 2026-11-14T17:00:06.126Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin


✅ Part 3 Scraping complete! Saved to scraped_prices_part3.csv


In [9]:
import pandas as pd

# 1. Load the main dataset
df = pd.read_csv("Data/kenya_resorts_clean.csv")

# 2. Load the last batch of scraped prices (Part 3)
part3_df = pd.read_csv("Data/scraped_prices_part3.csv")

# 3. Merge the new prices into the main dataset
# suffixes=('', '_new') ensures if a column exists in both, the new one gets a '_new' tag
final_df = pd.merge(df, part3_df, on="ResortName", how="left", suffixes=('', '_new'))

# 4. Fill missing values in the original columns with the newly scraped ones
if 'BasePrice' in final_df.columns:
    final_df['BasePrice'] = final_df['BasePrice'].fillna(final_df['BasePrice_new'])
    final_df['HotelClass'] = final_df['HotelClass'].fillna(final_df['HotelClass_new'])
    final_df['BasePrice_KES'] = final_df['BasePrice_KES'].fillna(final_df['BasePrice_KES_new'])

# 5. Drop the temporary '_new' columns so your dataset is clean
cols_to_drop = [c for c in final_df.columns if c.endswith('_new')]
final_df = final_df.drop(columns=cols_to_drop)

# 6. Ensure KES is calculated for all prices (USD * 150)
final_df['BasePrice_KES'] = final_df['BasePrice'] * 150

# 7. Save back to the exact same file Tableau uses
final_df.to_csv("Data/kenya_resorts_clean.csv", index=False)

print("✅ MERGE COMPLETE!")
print(f"Total Resorts with prices: {final_df['BasePrice'].notna().sum()} out of {len(final_df)}")

✅ MERGE COMPLETE!
Total Resorts with prices: 72 out of 168


In [17]:
import pandas as pd

# Load the file
df = pd.read_csv("Data/kenya_resorts_clean.csv")

# Print how many prices we have
print(f"Total Resorts with prices: {df['BasePrice'].notna().sum()} out of {len(df)}")
print(df[["ResortName", "BasePrice", "BasePrice_KES"]].dropna().head(10))

Total Resorts with prices: 72 out of 168
                              ResortName  BasePrice  BasePrice_KES
0          Diani Reef Beach Resort & Spa      150.0        22500.0
1                   Swahili Beach Resort      267.0        40050.0
2   Diamonds Leisure Beach & Golf Resort      180.0        27000.0
4            Southern Palms Beach Resort      199.0        29850.0
5    Jacaranda Indian Ocean Beach Resort      112.0        16800.0
6                   Kaskazi Beach Resort      106.0        15900.0
7          The Villa Luxury Suites Hotel       86.0        12900.0
8                            Tombo House       98.0        14700.0
9               Bahari Dhow Beach Villas       95.0        14250.0
10         The Maji Beach Boutique Hotel      328.0        49200.0


In [18]:
import os
import pandas as pd
from apify_client import ApifyClient
from dotenv import load_dotenv
import time

# 1. Load the token from your .env file
load_dotenv(override=True) 
APIFY_TOKEN = os.getenv("APIFY_API_TOKEN")
client = ApifyClient(APIFY_TOKEN)

# 2. Load your dataset
df = pd.read_csv("Data/kenya_resorts_clean.csv")

# 3. Find exactly which resorts are still missing a price
missing_df = df[df["BasePrice"].isna()]
print(f"⏳ Found {len(missing_df)} missing resorts. Scraping them with your remaining Apify credit...")

results = []

for index, row in missing_df.iterrows():
    resort_name = row["ResortName"]
    print(f"[{index+1}/168] Scraping: {resort_name}...")
    
    run_input = {
        "destination": f"{resort_name}, Kenya",
        "checkIn": "2026-12-01",  
        "checkOut": "2026-12-02",
        "adults": 2,
        "currency": "USD",
        "maxResultsPerSearch": 1 
    }
    
    try:
        run = client.actor("inexhaustible_glass/google-hotels-scraper").call(run_input=run_input)
        
        item_found = False
        for item in client.dataset(run.default_dataset_id).iterate_items():
            price = item.get("price")
            results.append({
                "ResortName": resort_name,
                "BasePrice": price,
                "HotelClass": item.get("hotel_class_label"),
                "BasePrice_KES": price * 150 if price else None
            })
            item_found = True
            break 
            
        if not item_found:
            results.append({"ResortName": resort_name, "BasePrice": None, "HotelClass": None, "BasePrice_KES": None})
            
        time.sleep(2) # Pause to respect limits
        
    except Exception as e:
        print(f"⚠️ Error on {resort_name}: {e}")
        results.append({"ResortName": resort_name, "BasePrice": None, "HotelClass": None, "BasePrice_KES": None})

# 4. Merge the new prices with the old data
new_apify_df = pd.DataFrame(results)
new_apify_df = new_apify_df.drop_duplicates(subset=["ResortName"])

final_df = pd.merge(df, new_apify_df, on="ResortName", how="left", suffixes=('', '_new'))

if 'BasePrice' in final_df.columns and 'BasePrice_new' in final_df.columns:
    final_df['BasePrice'] = final_df['BasePrice'].fillna(final_df['BasePrice_new'])
    final_df['HotelClass'] = final_df['HotelClass'].fillna(final_df['HotelClass_new'])
    final_df['BasePrice_KES'] = final_df['BasePrice_KES'].fillna(final_df['BasePrice_KES_new'])

# Clean up the temporary columns
cols_to_drop = [c for c in final_df.columns if c.endswith('_new')]
final_df = final_df.drop(columns=cols_to_drop)

# Recalculate KES for everything just to be safe
final_df['BasePrice_KES'] = final_df['BasePrice'] * 150

# 5. Save back to the exact same file Tableau uses
final_df.to_csv("Data/kenya_resorts_clean.csv", index=False)

print("\n✅ MERGE COMPLETE!")
print(f"Total Resorts with prices: {final_df['BasePrice'].notna().sum()} out of {len(final_df)}")

⏳ Found 96 missing resorts. Scraping them with your remaining Apify credit...
[4/168] Scraping: Coral Beach Resort...


[apify.google-hotels-scraper runId:GK26rNCBhILKQAQdQ] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:GK26rNCBhILKQAQdQ] -> 2026-11-14T17:30:17.673Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:GK26rNCBhILKQAQdQ] -> 2026-11-14T17:30:20.505Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:GK26rNCBhILKQAQdQ] -> 2026-11-14T17:30:20.576Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:GK26rNCBhILKQAQdQ] -> 2026-11-14T17:30:20.578Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:GK26rNCBhILKQAQdQ] -> 2026-11-14T17:30:21.487Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:GK26rNCBhILKQAQdQ] -> 2026-11-14T17:30:21.488Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

[14/168] Scraping: Aqua Resort...


[apify.google-hotels-scraper runId:ax2kx8yIje1ithcXA] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:ax2kx8yIje1ithcXA] -> 2026-11-14T17:30:33.250Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:ax2kx8yIje1ithcXA] -> 2026-11-14T17:30:35.278Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:ax2kx8yIje1ithcXA] -> 2026-11-14T17:30:35.457Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:ax2kx8yIje1ithcXA] -> 2026-11-14T17:30:35.458Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:ax2kx8yIje1ithcXA] -> 2026-11-14T17:30:36.536Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:ax2kx8yIje1ithcXA] -> 2026-11-14T17:30:36.538Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

[15/168] Scraping: Alfajiri Villas...


[apify.google-hotels-scraper runId:akxuAHymNhPzaSncp] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:akxuAHymNhPzaSncp] -> 2026-11-14T17:30:48.831Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:akxuAHymNhPzaSncp] -> 2026-11-14T17:30:48.836Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:akxuAHymNhPzaSncp] -> 2026-11-14T17:30:48.887Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:akxuAHymNhPzaSncp] -> 2026-11-14T17:30:48.888Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:akxuAHymNhPzaSncp] -> 2026-11-14T17:30:50.273Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:akxuAHymNhPzaSncp] -> 2026-11-14T17:30:50.274Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

... (36 outputs left out)

[47/168] Scraping: Baobab Village...


[apify.google-hotels-scraper runId:FznmHVpIEKOP7KcFo] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:FznmHVpIEKOP7KcFo] -> 2026-11-14T17:34:59.635Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:FznmHVpIEKOP7KcFo] -> 2026-11-14T17:34:59.637Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:FznmHVpIEKOP7KcFo] -> 2026-11-14T17:34:59.687Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:FznmHVpIEKOP7KcFo] -> 2026-11-14T17:34:59.688Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:FznmHVpIEKOP7KcFo] -> 2026-11-14T17:35:00.458Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:FznmHVpIEKOP7KcFo] -> 2026-11-14T17:35:00.460Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

[49/168] Scraping: Shela Bliss...
⚠️ Error on Shela Bliss: POST /v2/actors/inexhaustible_glass~google-hotels-scraper/runs: You've used the $5.00 of monthly usage included in the Free plan, so Actor runs and other Apify services are blocked until your monthly usage cycle resets. GET https://api.apify.com/v2/users/me/limits shows when (monthlyUsageCycle.endAt). The Free plan limit can't be raised. To continue now, upgrade at https://console.apify.com/billing/subscription.
[50/168] Scraping: Wiyoni Royals Lamu...
⚠️ Error on Wiyoni Royals Lamu: POST /v2/actors/inexhaustible_glass~google-hotels-scraper/runs: You've used the $5.00 of monthly usage included in the Free plan, so Actor runs and other Apify services are blocked until your monthly usage cycle resets. GET https://api.apify.com/v2/users/me/limits shows when (monthlyUsageCycle.endAt). The Free plan limit can't be raised. To continue now, upgrade at https://console.apify.com/billing/subscription.
[54/168] Scraping: Forodhani House -

C:\Users\User\AppData\Local\Temp\ipykernel_3436\3418142771.py:65: Pandas4Warning: 'ndarray' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  final_df['BasePrice'] = final_df['BasePrice'].fillna(final_df['BasePrice_new'])
C:\Users\User\AppData\Local\Temp\ipykernel_3436\3418142771.py:67: Pandas4Warning: 'ndarray' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  final_df['BasePrice_KES'] = final_df['BasePrice_KES'].fillna(final_df['BasePrice_KES_new'])


In [19]:
import pandas as pd

# 1. Load the original file
df = pd.read_csv("Data/kenya_resorts_clean.csv")

# 2. Create a dictionary of the prices we just scraped
new_prices = dict(zip(new_apify_df['ResortName'], new_apify_df['BasePrice']))
new_classes = dict(zip(new_apify_df['ResortName'], new_apify_df['HotelClass']))

# 3. Force update the dataframe directly
for idx, row in df.iterrows():
    name = row['ResortName']
    # If it's missing a price and we just scraped one for it:
    if pd.isna(row['BasePrice']) and name in new_prices:
        price = new_prices[name]
        if pd.notna(price):
            df.at[idx, 'BasePrice'] = price
            df.at[idx, 'BasePrice_KES'] = price * 150
            df.at[idx, 'HotelClass'] = new_classes[name]

# 4. Save to the file
df.to_csv("Data/kenya_resorts_clean.csv", index=False)

print("✅ FIXED AND SAVED!")
print(f"Total Resorts with prices: {df['BasePrice'].notna().sum()} out of {len(df)}")

✅ FIXED AND SAVED!
Total Resorts with prices: 72 out of 168


In [20]:
import pandas as pd

df = pd.read_csv("Data/kenya_resorts_clean.csv")

# Force update the 9 new prices from memory into the CSV
for idx, row in new_apify_df.iterrows():
    if pd.notna(row['BasePrice']):
        df_idx = df[df['ResortName'] == row['ResortName']].index
        if len(df_idx) > 0:
            df.loc[df_idx, 'BasePrice'] = row['BasePrice']
            df.loc[df_idx, 'HotelClass'] = row['HotelClass']
            df.loc[df_idx, 'BasePrice_KES'] = row['BasePrice'] * 150

df.to_csv("Data/kenya_resorts_clean.csv", index=False)
print(f"✅ Saved. Total Resorts with prices: {df['BasePrice'].notna().sum()} out of {len(df)}")

✅ Saved. Total Resorts with prices: 72 out of 168


In [23]:
import os
import pandas as pd
from apify_client import ApifyClient
from dotenv import load_dotenv
import time

# 1. Load the NEW token
load_dotenv(override=True) 
client = ApifyClient(os.getenv("APIFY_API_TOKEN"))

# 2. Load dataset
df = pd.read_csv("Data/kenya_resorts_clean.csv")

# 3. Find missing
missing_df = df[df["BasePrice"].isna()]
print(f"⏳ Found {len(missing_df)} missing resorts. Scraping with new account...")

# 4. Scrape and update directly
for index, row in missing_df.iterrows():
    resort_name = row["ResortName"]
    print(f"[{index+1}/168] Scraping: {resort_name}...")
    
    run_input = {
        "destination": f"{resort_name}, Kenya",
        "checkIn": "2026-12-01",  
        "checkOut": "2026-12-02",
        "adults": 2,
        "currency": "USD",
        "maxResultsPerSearch": 1 
    }
    
    try:
        run = client.actor("inexhaustible_glass/google-hotels-scraper").call(run_input=run_input)
        
        for item in client.dataset(run.default_dataset_id).iterate_items():
            price = item.get("price")
            if price:
                df.at[index, "BasePrice"] = price
                df.at[index, "HotelClass"] = item.get("hotel_class_label")
                df.at[index, "BasePrice_KES"] = price * 150
                print(f"   ✅ Found: ${price}")
            break 
            
        time.sleep(2) 
        
    except Exception as e:
        print(f"   ⚠️ Hit limit or error.")
        break 

# 5. Save
df.to_csv("Data/kenya_resorts_clean.csv", index=False)

print("\n✅ DONE!")
print(f"Total Resorts with prices: {df['BasePrice'].notna().sum()} out of {len(df)}")

⏳ Found 70 missing resorts. Scraping with new account...
[4/168] Scraping: Coral Beach Resort...


[apify.google-hotels-scraper runId:G9jGSYzwODX4RWkDj] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:G9jGSYzwODX4RWkDj] -> 2026-11-14T18:44:32.727Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:G9jGSYzwODX4RWkDj] -> 2026-11-14T18:44:32.729Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:G9jGSYzwODX4RWkDj] -> 2026-11-14T18:44:32.780Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:G9jGSYzwODX4RWkDj] -> 2026-11-14T18:44:32.781Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:G9jGSYzwODX4RWkDj] -> 2026-11-14T18:44:33.994Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:G9jGSYzwODX4RWkDj] -> 2026-11-14T18:44:33.995Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

[14/168] Scraping: Aqua Resort...


[apify.google-hotels-scraper runId:broXFo21cD3YNILZW] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:broXFo21cD3YNILZW] -> 2026-11-14T18:44:46.921Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:broXFo21cD3YNILZW] -> 2026-11-14T18:44:46.923Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:broXFo21cD3YNILZW] -> 2026-11-14T18:44:46.983Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:broXFo21cD3YNILZW] -> 2026-11-14T18:44:46.983Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:broXFo21cD3YNILZW] -> 2026-11-14T18:44:48.302Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:broXFo21cD3YNILZW] -> 2026-11-14T18:44:48.303Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

[15/168] Scraping: Alfajiri Villas...


[apify.google-hotels-scraper runId:mzrmKui8fBsgypQsw] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:mzrmKui8fBsgypQsw] -> 2026-11-14T18:45:01.103Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:mzrmKui8fBsgypQsw] -> 2026-11-14T18:45:01.106Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:mzrmKui8fBsgypQsw] -> 2026-11-14T18:45:01.155Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:mzrmKui8fBsgypQsw] -> 2026-11-14T18:45:01.156Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:mzrmKui8fBsgypQsw] -> 2026-11-14T18:45:01.983Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:mzrmKui8fBsgypQsw] -> 2026-11-14T18:45:01.984Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

... (119 outputs left out)

[apify.google-hotels-scraper runId:5ffGMLyi5nYk1IIIG] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:5ffGMLyi5nYk1IIIG] -> 2026-11-14T18:57:59.501Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:5ffGMLyi5nYk1IIIG] -> 2026-11-14T18:57:59.504Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:5ffGMLyi5nYk1IIIG] -> 2026-11-14T18:57:59.558Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:5ffGMLyi5nYk1IIIG] -> 2026-11-14T18:57:59.560Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:5ffGMLyi5nYk1IIIG] -> 2026-11-14T18:58:00.359Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:5ffGMLyi5nYk1IIIG] -> 2026-11-14T18:58:00.361Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

[158/168] Scraping: Royal Mara Safari Lodge...


[apify.google-hotels-scraper runId:AFdMoqtTvfDdwgGvK] -> Status: RUNNING, Message: 
[apify.google-hotels-scraper runId:AFdMoqtTvfDdwgGvK] -> 2026-11-14T18:58:11.181Z ACTOR: Pulling container image of build o0V82H0NDV2i7g10r from registry.
[apify.google-hotels-scraper runId:AFdMoqtTvfDdwgGvK] -> 2026-11-14T18:58:11.184Z ACTOR: Creating container.
[apify.google-hotels-scraper runId:AFdMoqtTvfDdwgGvK] -> 2026-11-14T18:58:11.240Z ACTOR: Starting container.
[apify.google-hotels-scraper runId:AFdMoqtTvfDdwgGvK] -> 2026-11-14T18:58:11.241Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.google-hotels-scraper runId:AFdMoqtTvfDdwgGvK] -> 2026-11-14T18:58:12.171Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.google-hotels-scraper runId:AFdMoqtTvfDdwgGvK] -> 2026-11-14T18:58:12.182Z [apify] INFO  Initializing Actor ({"apify_sdk_version": "4.1.0", "apify_client_version": "3.3.0", "crawlee_version": "1.11.0", "python_version": "3.12.15", "os": "lin

[159/168] Scraping: Karen Blixen Camp, Mara North Conservancy, Masai Mara...
   ⚠️ Hit limit or error.

✅ DONE!
Total Resorts with prices: 101 out of 168


In [24]:
import pandas as pd

df = pd.read_csv("Data/kenya_resorts_clean.csv")

# Count how many resorts have a price, grouped by County
price_counts = df.groupby("County")["BasePrice"].count()
total_counts = df.groupby("County")["ResortName"].count()

# Combine them into one table
summary = pd.DataFrame({
    "Total Resorts": total_counts,
    "Resorts with Price": price_counts,
    "Missing Prices": total_counts - price_counts
})

print(summary)

         Total Resorts  Resorts with Price  Missing Prices
County                                                    
Isiolo               1                   0               1
Kajiado              2                   0               2
Kilifi              20                   6              14
Kisumu              20                  14               6
Kwale               20                  15               5
Lamu                20                   8              12
Mombasa              7                   5               2
Nakuru              59                  41              18
Narok               19                  12               7


In [26]:
import pandas as pd

# 1. Load the dataset (this currently has the fake imputed prices)
df = pd.read_csv("Data/kenya_resorts_clean.csv")

# 2. We need to restore the real prices. 
# We know exactly which resorts have real prices by merging the scraped parts again.
# But the easiest way is just to drop the imputed ones and save.

# Actually, let's just tell Tableau to ignore the nulls.
# Run this to verify how many real prices we have before saving.
print(f"Real Prices: {df['BasePrice'].notna().sum()}")

# If this still says 168, it means the imputation is still in there.
# We will fix this in Tableau by filtering out the fake values, 
# but for now, just delete the imputation cell so it doesn't run again.

Real Prices: 168
